# 🏁 F1 Podium Predictor — 77 seasons of Formula 1 (1950 → Brazil 2026)
**Hussain Ahmed** · Data & Machine Learning

**Data:** *Formula 1 World Championship (1950 - 2026)* by Samuel_Pbq on Kaggle — Jolpica-F1 / Ergast tables,
1,169 races up to the 2026 Brazilian GP (round 20). License **CC BY-NC-SA 4.0** (non-commercial, share-alike).

**Plan**
1. **Analysis** — what 77 seasons say about poles, reliability, pit stops, circuits, dominance and the new 2026 rules.
2. **Model** — *before the lights go out*, what is each driver's probability of finishing on the podium?
   Leak-free features only (Saturday's grid + everything before the race), season-forward validation,
   Optuna tuning, held-out test on **2023–2026**.
3. **Explain** — SHAP, calibration, and race-level case studies.

## 0. Setup for Kaggle
The project's helper modules are written to `/kaggle/working/src` so this notebook is self-contained. Full repo (dashboard + Streamlit app): https://github.com/HussainAhmed75/f1-podium-predictor

In [ ]:
!mkdir -p /kaggle/working/src /kaggle/working/figures /kaggle/working/results /kaggle/working/models /kaggle/working/data
!pip install -q optuna shap 2>/dev/null | tail -1

In [ ]:
%%writefile /kaggle/working/src/features.py
"""Leak-free, pre-race feature engineering for F1 podium prediction.

Every feature for race R uses only information available *before the race starts*:
- qualifying / starting grid of race R itself (known on Saturday),
- results and standings of races strictly before R.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

RAW = "data/raw/"
CLASSIFIED_RE = r"^(Finished|Lapped|\+\d+ Laps?)$"
START_YEAR = 2006  # qualifying is complete from 2003; 3 seasons of warm-up for rolling stats


def _time_to_sec(s: pd.Series) -> pd.Series:
    """'1:23.456' -> 83.456 ; NaN stays NaN."""
    parts = s.astype("string").str.extract(r"^(?:(\d+):)?(\d+\.\d+)$")
    mins = pd.to_numeric(parts[0], errors="coerce").fillna(0)
    secs = pd.to_numeric(parts[1], errors="coerce")
    return mins * 60 + secs


def load_tables(raw: str = RAW) -> dict[str, pd.DataFrame]:
    names = ["races", "results", "status", "drivers", "constructors", "circuits", "qualifying",
             "driver_standings", "constructor_standings", "pit_stops", "sprint_results"]
    return {n: pd.read_csv(f"{raw}{n}.csv") for n in names}


def base_results(t: dict[str, pd.DataFrame]) -> pd.DataFrame:
    races = t["races"][["raceId", "year", "round", "circuitId", "name", "date", "sprint_date"]].copy()
    races["date"] = pd.to_datetime(races["date"])
    df = (t["results"].merge(races, on="raceId").merge(t["status"], on="statusId")
          .sort_values(["date", "raceId", "positionOrder"]).reset_index(drop=True))
    df["dns"] = df["status"].isin(["Did not start", "Withdrew", "Did not qualify", "Did not prequalify"])
    df["classified"] = df["status"].str.match(CLASSIFIED_RE)
    df["dnf"] = ((~df["classified"]) & (~df["dns"])).astype(float)
    df["podium"] = (df["positionOrder"] <= 3).astype(int)
    df["win"] = (df["positionOrder"] == 1).astype(int)
    return df


def _rolling_prev(g: pd.core.groupby.SeriesGroupBy, window: int, how: str = "mean") -> pd.Series:
    """Rolling stat over the previous `window` rows of each group (current row excluded)."""
    return g.transform(lambda s: getattr(s.shift(1).rolling(window, min_periods=1), how)())


def build_features(t: dict[str, pd.DataFrame], start_year: int = START_YEAR) -> pd.DataFrame:
    df = base_results(t)
    df = df[~df["dns"]].copy()

    # ---------- grid & qualifying (known before the race) ----------
    n_cars = df.groupby("raceId")["driverId"].transform("size")
    df["grid_pos"] = np.where(df["grid"] > 0, df["grid"], n_cars)  # 0 = pit-lane start -> back of grid
    df["pitlane_start"] = (df["grid"] == 0).astype(int)
    df["n_cars"] = n_cars

    q = t["qualifying"].copy()
    for c in ["q1", "q2", "q3"]:
        q[c + "_s"] = _time_to_sec(q[c])
    q["best_q"] = q[["q1_s", "q2_s", "q3_s"]].min(axis=1)
    q["pole_q"] = q.groupby("raceId")["best_q"].transform("min")
    q["quali_gap_pct"] = 100 * (q["best_q"] / q["pole_q"] - 1)
    q["reached_q3"] = q["q3_s"].notna().astype(int)
    df = df.merge(q[["raceId", "driverId", "position", "quali_gap_pct", "reached_q3"]]
                  .rename(columns={"position": "quali_pos"}), on=["raceId", "driverId"], how="left")
    df.loc[df["quali_gap_pct"] > 15, "quali_gap_pct"] = np.nan  # in/out-lap artefacts

    # teammate comparison on the grid
    team_best = df.groupby(["raceId", "constructorId"])["grid_pos"].transform("min")
    team_worst = df.groupby(["raceId", "constructorId"])["grid_pos"].transform("max")
    df["grid_vs_teammate"] = np.where(df["grid_pos"] == team_best, df["grid_pos"] - team_worst,
                                      df["grid_pos"] - team_best)
    df["sprint_weekend"] = df["sprint_date"].notna().astype(int)
    df["round_frac"] = df["round"] / df.groupby("year")["round"].transform("max")

    # ---------- driver form (previous races only) ----------
    df = df.sort_values(["date", "raceId"]).reset_index(drop=True)
    gd = df.groupby("driverId")
    df["drv_avg_finish_5"] = _rolling_prev(gd["positionOrder"], 5)
    df["drv_podium_rate_10"] = _rolling_prev(gd["podium"], 10)
    df["drv_win_rate_10"] = _rolling_prev(gd["win"], 10)
    df["drv_dnf_rate_10"] = _rolling_prev(gd["dnf"], 10)
    df["drv_avg_grid_5"] = _rolling_prev(gd["grid_pos"], 5)
    df["drv_gain_5"] = _rolling_prev(gd["grid_pos"], 5) - df["drv_avg_finish_5"]  # avg places gained
    df["drv_experience"] = gd.cumcount()
    df["drv_career_podiums"] = gd["podium"].transform(lambda s: s.shift(1).cumsum()).fillna(0)

    # ---------- constructor form (previous races only) ----------
    team_race = (df.groupby(["raceId", "date", "constructorId"])
                 .agg(team_best=("positionOrder", "min"), team_pod=("podium", "max"),
                      team_dnf=("dnf", "mean"), team_grid=("grid_pos", "min"))
                 .reset_index().sort_values(["date", "raceId"]))
    gt = team_race.groupby("constructorId")
    team_race["team_best_finish_5"] = _rolling_prev(gt["team_best"], 5)
    team_race["team_podium_rate_10"] = _rolling_prev(gt["team_pod"], 10)
    team_race["team_dnf_rate_10"] = _rolling_prev(gt["team_dnf"], 10)
    team_race["team_best_grid_5"] = _rolling_prev(gt["team_grid"], 5)
    df = df.merge(team_race[["raceId", "constructorId", "team_best_finish_5", "team_podium_rate_10",
                             "team_dnf_rate_10", "team_best_grid_5"]], on=["raceId", "constructorId"], how="left")

    # ---------- championship standings after the previous round of the same season ----------
    races = t["races"][["raceId", "year", "round"]]
    nxt = races.copy(); nxt["round"] += 1  # standings after round r are "before round r+1"
    ds = t["driver_standings"].merge(nxt, on="raceId")[["year", "round", "driverId", "position", "points"]]
    ds["leader_pts"] = ds.groupby(["year", "round"])["points"].transform("max")
    ds["drv_pts_share_of_leader"] = np.where(ds["leader_pts"] > 0, ds["points"] / ds["leader_pts"], 0)
    df = df.merge(ds.rename(columns={"position": "drv_champ_pos"})
                  [["year", "round", "driverId", "drv_champ_pos", "drv_pts_share_of_leader"]],
                  on=["year", "round", "driverId"], how="left")
    cs = t["constructor_standings"].merge(nxt, on="raceId")[["year", "round", "constructorId", "position"]]
    df = df.merge(cs.rename(columns={"position": "team_champ_pos"}), on=["year", "round", "constructorId"], how="left")

    # ---------- circuit history (previous visits only) ----------
    df = df.sort_values(["date", "raceId"]).reset_index(drop=True)
    gdc = df.groupby(["driverId", "circuitId"])
    df["drv_circuit_avg_finish"] = gdc["positionOrder"].transform(lambda s: s.shift(1).expanding().mean())
    df["drv_circuit_podiums"] = gdc["podium"].transform(lambda s: s.shift(1).cumsum()).fillna(0)
    # how much the grid order is shuffled at this circuit, from earlier races there
    df["abs_moves"] = (df["grid_pos"] - df["positionOrder"]).abs().where(df["classified"])
    circ_race = df.groupby(["raceId", "date", "circuitId"])["abs_moves"].mean().reset_index().sort_values("date")
    circ_race["circuit_shuffle"] = circ_race.groupby("circuitId")["abs_moves"].transform(
        lambda s: s.shift(1).expanding().mean())
    df = df.merge(circ_race[["raceId", "circuit_shuffle"]], on="raceId", how="left")

    df = df[df["year"] >= start_year].reset_index(drop=True)
    return df


FEATURES = [
    "grid_pos", "pitlane_start", "quali_pos", "quali_gap_pct", "reached_q3", "grid_vs_teammate", "n_cars",
    "sprint_weekend", "round_frac",
    "drv_avg_finish_5", "drv_podium_rate_10", "drv_win_rate_10", "drv_dnf_rate_10", "drv_avg_grid_5",
    "drv_gain_5", "drv_experience", "drv_career_podiums",
    "team_best_finish_5", "team_podium_rate_10", "team_dnf_rate_10", "team_best_grid_5",
    "drv_champ_pos", "drv_pts_share_of_leader", "team_champ_pos",
    "drv_circuit_avg_finish", "drv_circuit_podiums", "circuit_shuffle",
]
TARGET = "podium"


In [ ]:
%%writefile /kaggle/working/src/modeling.py
"""Model zoo, season-forward cross-validation and race-level evaluation helpers."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SEED = 42
TEST_YEARS = [2023, 2024, 2025, 2026]
CV_YEARS = [2018, 2019, 2020, 2021, 2022]  # each validated on a model trained on all earlier seasons


def make_models() -> dict:
    return {
        "Grid-only logistic (baseline)": ("grid", Pipeline([("sc", StandardScaler()),
                                                              ("m", LogisticRegression(max_iter=2000))])),
        "Logistic regression": ("all", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                                                 ("sc", StandardScaler()),
                                                 ("m", LogisticRegression(C=0.5, max_iter=5000))])),
        "Random forest": ("all", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                                           ("m", RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                                                        n_jobs=-1, random_state=SEED))])),
        "XGBoost": ("all", XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=4, subsample=0.8,
                                         colsample_bytree=0.8, eval_metric="logloss", random_state=SEED, n_jobs=4)),
        "LightGBM": ("all", LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, subsample=0.8,
                                           subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)),
    }


def race_top3_hits(frame: pd.DataFrame, score: np.ndarray) -> pd.Series:
    """For each race, pick the 3 highest-scored drivers; return how many actually finished on the podium."""
    f = frame[["raceId", "podium"]].copy()
    f["score"] = score
    f["rank"] = f.groupby("raceId")["score"].rank(ascending=False, method="first")
    return f[f["rank"] <= 3].groupby("raceId")["podium"].sum()


def race_winner_hit(frame: pd.DataFrame, score: np.ndarray) -> float:
    f = frame[["raceId", "win"]].copy()
    f["score"] = score
    top = f.loc[f.groupby("raceId")["score"].idxmax()]
    return float(top["win"].mean())


def metrics(frame: pd.DataFrame, y: np.ndarray, p: np.ndarray) -> dict:
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "log_loss": log_loss(y, p), "brier": brier_score_loss(y, p),
            "podium_hits_per_race": float(race_top3_hits(frame, p).mean()),
            "winner_hit_rate": race_winner_hit(frame, p)}


def season_forward_cv(model, df: pd.DataFrame, cols: list[str], target: str = "podium",
                      years: list[int] = CV_YEARS) -> pd.DataFrame:
    rows = []
    for yr in years:
        tr, va = df[df["year"] < yr], df[df["year"] == yr]
        m = clone(model).fit(tr[cols], tr[target])
        p = m.predict_proba(va[cols])[:, 1]
        rows.append({"year": yr, **metrics(va, va[target].values, p)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /kaggle/working/src/style.py
"""One visual system for every chart in the project (dark, F1-style)."""
import matplotlib.pyplot as plt

SURFACE = "#1a1a19"
INK = "#f0efec"
INK_2 = "#c3c2b7"
MUTED = "#8a8980"
GRID = "#33332f"
BLUE = "#3987e5"     # series 1 (dark-mode step)
ORANGE = "#d95926"   # series 2 / highlight
AQUA = "#199e70"     # series 3
GRAY = "#55554f"     # de-emphasised marks
SERIES = [BLUE, ORANGE, AQUA, "#c98500", "#d55181"]


def apply():
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
        "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
        "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 26,
        "font.size": 11, "font.family": "DejaVu Sans", "legend.frameon": False, "lines.linewidth": 2,
        "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
    })


def subtitle(ax, text):
    ax.text(0, 1.02, text, transform=ax.transAxes, color=INK_2, fontsize=10.5, va="bottom")


def source(fig, text="Data: Jolpica-F1 / Ergast via Kaggle (Samuel_Pbq), CC BY-NC-SA 4.0 · Hussain Ahmed"):
    fig.text(0.01, -0.02, text, color=MUTED, fontsize=8.5, ha="left", va="top")


In [ ]:
import sys, json, warnings
sys.path.insert(0, "/kaggle/working/src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import optuna, shap, joblib
from lightgbm import LGBMClassifier
from sklearn.calibration import calibration_curve

import style
from style import BLUE, ORANGE, AQUA, GRAY, INK, INK_2, MUTED
from features import load_tables, base_results, build_features, FEATURES, TARGET, RAW
from modeling import make_models, season_forward_cv, metrics, race_top3_hits, race_winner_hit, TEST_YEARS, CV_YEARS, SEED

import glob, os
RAW_DIR = os.path.dirname(glob.glob("/kaggle/input/**/races.csv", recursive=True)[0]) + "/"
FIG, RES = "/kaggle/working/figures/", "/kaggle/working/results/"
style.apply()
pd.set_option("display.width", 160)
facts = {}

## 1. Load & sanity-check

In [ ]:
t = load_tables(RAW_DIR)
for k, v in t.items():
    print(f"{k:24s} {v.shape}")
res = base_results(t)
facts["n_races"] = int(t["races"].raceId.nunique()); facts["n_seasons"] = int(t["races"].year.nunique())
facts["last_race"] = t["races"].sort_values("raceId").iloc[-1][["year", "round", "name", "date"]].astype(str).to_dict()
winners_per_race = res.groupby("raceId").positionOrder.apply(lambda s: (s == 1).sum())
print("races with a shared win (1950s shared drives, documented in the source):", (winners_per_race != 1).sum())
assert winners_per_race[res.groupby("raceId").year.first() >= 1960].eq(1).all()
facts

**Classification rule** — `status` is *Finished*, *Lapped* or *+N Laps* → classified; *Did not start / Withdrew / DNQ*
→ removed; anything else (Retired, Accident, Engine, …) → **DNF**.

## 2. Analysis — 7 stories in the data

### 2.1 Pole position is worth more than ever

In [ ]:
wins = res[res.positionOrder == 1].copy()
wins["decade"] = wins.year // 10 * 10
pole = (wins.grid == 1).groupby(wins.decade).mean()
facts["pole_win_rate_by_decade"] = pole.round(3).to_dict()
fig, ax = plt.subplots(figsize=(9, 4.6))
bars = ax.bar(pole.index.astype(str) + "s", pole.values * 100, color=[GRAY] * (len(pole) - 1) + [ORANGE], width=0.62)
for b, v in zip(bars, pole.values):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 1.2, f"{v:.0%}", ha="center", color=INK, fontsize=11)
ax.set_ylim(0, 70); ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.grid(axis="x", visible=False)
ax.set_title("Pole position wins more often than ever")
style.subtitle(ax, "Share of Grands Prix won from pole, by decade")
style.source(fig); plt.savefig(FIG + "01_pole_to_win.png"); plt.show()

### 2.2 Since 2010, the grid slot almost decides the podium

In [ ]:
mod = res[(res.year >= 2010) & ~res.dns & (res.grid > 0)]
pg = mod.groupby("grid").podium.mean().loc[1:12]
facts["p_podium_by_grid_2010plus"] = pg.round(3).to_dict()
fig, ax = plt.subplots(figsize=(9, 4.6))
cols = [ORANGE if g <= 3 else BLUE for g in pg.index]
bars = ax.bar(pg.index.astype(str), pg.values * 100, color=cols, width=0.62)
for b, v in zip(bars, pg.values):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 1.2, f"{v:.0%}", ha="center", color=INK, fontsize=10)
ax.set_xlabel("Starting grid position"); ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.grid(axis="x", visible=False)
ax.set_title("Where you start ≈ where you finish")
style.subtitle(ax, "Chance of a podium by grid slot, 2010–2026")
style.source(fig); plt.savefig(FIG + "02_podium_by_grid.png"); plt.show()

### 2.3 The greatest comebacks — and one is from 2026

In [ ]:
far = wins.sort_values("grid", ascending=False).head(8).merge(t["drivers"][["driverId", "forename", "surname"]], on="driverId")
far = far[["year", "name", "grid", "forename", "surname"]]
facts["furthest_back_winners"] = far.head(5).astype(str).to_dict(orient="records")
far

### 2.4 Reliability: 2026's new rules brought the retirements back

In [ ]:
starters = res[~res.dns]
dnf_year = starters.groupby("year").dnf.mean()
facts["dnf_rate"] = {int(y): round(float(v), 3) for y, v in dnf_year.loc[[1960, 1980, 2000, 2010, 2019, 2024, 2025, 2026]].items()}
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(dnf_year.index, dnf_year.values * 100, color=BLUE)
ax.scatter([2026], [dnf_year.loc[2026] * 100], color=ORANGE, s=70, zorder=3, edgecolor=style.SURFACE, linewidth=2)
low = dnf_year.loc[2000:].idxmin()
ax.annotate(f"2026: {dnf_year.loc[2026]:.0%}", (2026, dnf_year.loc[2026] * 100), xytext=(-70, 28), textcoords="offset points",
            color=INK, fontsize=11, arrowprops=dict(arrowstyle="-", color=MUTED))
ax.annotate(f"{low}: {dnf_year.loc[low]:.0%} (lowest)", (low, dnf_year.loc[low] * 100), xytext=(-40, -34),
            textcoords="offset points", color=INK_2, fontsize=10, arrowprops=dict(arrowstyle="-", color=MUTED))
ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.set_ylim(0, 75)
ax.set_title("Cars got bulletproof — until the 2026 regulations")
style.subtitle(ax, "Share of starters that did not finish (DNF), per season")
style.source(fig); plt.savefig(FIG + "03_dnf_rate.png"); plt.show()
facts["dnf_lowest_year_since_2000"] = int(low); facts["dnf_lowest_value"] = round(float(dnf_year.loc[low]), 3)

### 2.5 Pit stops: the crew got faster, the pit lane did not

In [ ]:
pit = t["pit_stops"].merge(t["races"][["raceId", "year"]], on="raceId")
pit["sec"] = pit.milliseconds / 1000
pit = pit[pit.sec < 60]           # removes red-flag 'stops' that last minutes
pit_med = pit.groupby("year").sec.median()
facts["pit_median_sec"] = pit_med.round(2).to_dict()
facts["pit_stops_used"] = int(len(pit))
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.plot(pit_med.index, pit_med.values, color=AQUA, marker="o", markersize=6)
ax.set_ylim(20, 26); ax.set_ylabel("seconds (pit-lane entry → exit)")
for y in [2011, 2014, 2026]:
    ax.annotate(f"{pit_med.loc[y]:.1f}s", (y, pit_med.loc[y]), xytext=(0, 10), textcoords="offset points", ha="center", color=INK)
ax.set_title("A pit stop costs ~23 seconds — every year")
style.subtitle(ax, "Median pit-lane time per stop (stops under 60 s), 2011–2026")
style.source(fig); plt.savefig(FIG + "04_pit_stops.png"); plt.show()

### 2.6 Where pole matters most (and least)

In [ ]:
cw = wins[wins.year >= 2000].merge(t["circuits"][["circuitId", "name"]].rename(columns={"name": "circuit"}), on="circuitId")
circ = cw.groupby("circuit").agg(races=("grid", "size"), pole_win=("grid", lambda s: (s == 1).mean())).query("races >= 15")
circ = circ.sort_values("pole_win")
facts["circuit_pole_win"] = circ.round(3).reset_index().to_dict(orient="records")
fig, ax = plt.subplots(figsize=(9, 6.2))
c = [ORANGE if i >= len(circ) - 3 else BLUE if i < 3 else GRAY for i in range(len(circ))]
ax.barh(circ.index, circ.pole_win * 100, color=c, height=0.62)
for i, (v, n) in enumerate(zip(circ.pole_win, circ.races)):
    ax.text(v * 100 + 1, i, f"{v:.0%}  ({n} races)", va="center", color=INK_2, fontsize=9.5)
ax.xaxis.set_major_formatter(mtick.PercentFormatter()); ax.set_xlim(0, 95); ax.grid(axis="y", visible=False)
ax.set_title("Pole is gold at some tracks, a coin flip at others")
style.subtitle(ax, "Share of races won from pole, 2000–2026 (circuits with 15+ races)")
style.source(fig); plt.savefig(FIG + "05_circuits_pole.png"); plt.show()

### 2.7 Dominance — and the 2026 title fight

In [ ]:
cwin = wins.groupby(["year", "constructorId"]).size().rename("w").reset_index()
cwin["share"] = cwin.w / cwin.year.map(wins.groupby("year").size())
top = cwin.loc[cwin.groupby("year").share.idxmax()].merge(t["constructors"][["constructorId", "name"]], on="constructorId")
facts["most_dominant_seasons"] = top.sort_values("share", ascending=False).head(5)[["year", "name", "w", "share"]].round(3).to_dict(orient="records")
top.sort_values("share", ascending=False).head(6)

In [ ]:
ds = t["driver_standings"].merge(t["races"][["raceId", "year", "round"]], on="raceId")
s26 = ds[ds.year == 2026].merge(t["drivers"][["driverId", "surname"]], on="driverId")
last = s26[s26["round"] == s26["round"].max()].sort_values("position")
lead5 = last.head(5).surname.tolist()
facts["standings_2026_latest"] = last.head(5)[["surname", "points", "wins"]].to_dict(orient="records")
facts["winners_2026"] = res[(res.year == 2026) & (res.positionOrder == 1)].merge(t["drivers"], on="driverId").surname.value_counts().to_dict()
fig, ax = plt.subplots(figsize=(10, 5))
ends = []
for i, name in enumerate(lead5):
    d = s26[s26.surname == name].sort_values("round")
    ax.plot(d["round"], d.points, color=style.SERIES[i], marker="o", markersize=4)
    ends.append([d.points.iloc[-1], name, style.SERIES[i], d["round"].iloc[-1]])
# spread end labels so they never overlap (min 16 points apart)
ends.sort(key=lambda e: -e[0]); ys = []
for e in ends:
    y = e[0] if not ys else min(e[0], ys[-1] - 16); ys.append(y)
for (pts, name, col, rnd), y in zip(ends, ys):
    ax.text(rnd + 0.4, y, f"{name}  {pts:.0f}", color=col, va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0.5, s26["round"].max() + 4); ax.set_xlabel("Round"); ax.set_ylabel("Championship points")
ax.set_title("2026: a rookie-season title charge")
style.subtitle(ax, "Drivers' championship points after each round (top 5 after the Brazilian GP)")
style.source(fig); plt.savefig(FIG + "06_title_fight_2026.png"); plt.show()

## 3. Model — podium probability before the race
**Unit:** one row per driver per race, 2006–2026 (2003–2005 only warm up the rolling statistics).

**Features (27), all known before the start:** grid & qualifying (position, % gap to pole, Q3 reached, gap to team-mate),
driver form (last 5/10 races: average finish, podium/win/DNF rate, places gained, experience, career podiums),
team form (best finish, podium & DNF rate), championship position and points share *after the previous round*,
circuit history (driver's past results there, how much the grid usually shuffles there), sprint weekend, season stage.
All rolling stats use `shift(1)` so the current race never leaks into its own features.

**Validation:** season-forward — for each season 2018–2022, train on every earlier season and predict that season.
**Test:** 2023–2026 (90 races), untouched until the end.

In [ ]:
df = build_features(t)
dev, test = df[~df.year.isin(TEST_YEARS)], df[df.year.isin(TEST_YEARS)]
print(df.shape, "| dev:", dev.raceId.nunique(), "races | test:", test.raceId.nunique(), "races | podium rate:", round(df[TARGET].mean(), 3))
assert dev.year.max() < min(TEST_YEARS)
facts["model_rows"] = int(len(df)); facts["dev_races"] = int(dev.raceId.nunique()); facts["test_races"] = int(test.raceId.nunique())
df[FEATURES].describe().T.round(2)

### 3.1 Compare models (season-forward CV, 2018–2022)

In [ ]:
rows, folds = [], {}
for name, (fs, model) in make_models().items():
    cols = ["grid_pos"] if fs == "grid" else FEATURES
    cv = season_forward_cv(model, dev, cols)
    folds[name] = cv
    rows.append({"model": name, **cv.drop(columns="year").mean().to_dict()})
# a model-free rule: the top 3 on the grid are the podium
r = [(race_top3_hits(dev[dev.year == y], -dev[dev.year == y].grid_pos.values).mean(),
      race_winner_hit(dev[dev.year == y], -dev[dev.year == y].grid_pos.values)) for y in CV_YEARS]
rows.append({"model": "Rule: top-3 on the grid", "podium_hits_per_race": np.mean(r, axis=0)[0], "winner_hit_rate": np.mean(r, axis=0)[1]})
cv_table = pd.DataFrame(rows)
cv_table.round(4)

### 3.2 Tune the strongest family with Optuna (objective = season-forward CV log-loss)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
def lgbm_from(p):
    return LGBMClassifier(n_estimators=p["n_estimators"], learning_rate=p["learning_rate"], num_leaves=p["num_leaves"],
                          min_child_samples=p["min_child_samples"], reg_lambda=p["reg_lambda"],
                          colsample_bytree=p["colsample_bytree"], subsample=p["subsample"], subsample_freq=1,
                          random_state=SEED, verbose=-1)
def objective(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
         "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
         "num_leaves": trial.suggest_int("num_leaves", 4, 31),
         "min_child_samples": trial.suggest_int("min_child_samples", 10, 200, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
         "subsample": trial.suggest_float("subsample", 0.5, 1.0)}
    return season_forward_cv(lgbm_from(p), dev, FEATURES).log_loss.mean()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=40)
best_params = study.best_params
tuned_cv = season_forward_cv(lgbm_from(best_params), dev, FEATURES)
cv_table = pd.concat([cv_table, pd.DataFrame([{"model": "LightGBM (Optuna-tuned)", **tuned_cv.drop(columns="year").mean().to_dict()}])],
                     ignore_index=True)
print(best_params)
cv_table.sort_values("log_loss").round(4)

In [ ]:
plot = cv_table.dropna(subset=["log_loss"]).sort_values("log_loss", ascending=False)
fig, ax = plt.subplots(figsize=(9, 4.2))
c = [ORANGE if "tuned" in m else GRAY if "baseline" in m else BLUE for m in plot.model]
ax.barh(plot.model, plot.log_loss, color=c, height=0.6)
for i, v in enumerate(plot.log_loss):
    ax.text(v + 0.002, i, f"{v:.4f}", va="center", color=INK_2, fontsize=10)
ax.set_xlim(0.18, 0.26); ax.grid(axis="y", visible=False); ax.set_xlabel("log-loss (lower = better probabilities)")
ax.set_title("Tuned LightGBM gives the sharpest probabilities")
style.subtitle(ax, "Season-forward CV on 2018–2022 (each season predicted by a model trained on earlier seasons)")
style.source(fig); plt.savefig(FIG + "07_model_comparison.png"); plt.show()

**Why tuned LightGBM?** Lowest CV log-loss and Brier score and highest ROC-AUC/PR-AUC of all candidates. Optuna chose a
*small, heavily regularised* model (4 leaves, ≥139 rows per leaf, slow learning rate): with ~20 cars per race
the signal is mostly a few strong effects (grid, team form), so simple trees generalise best.

## 4. Final test — 2023 to 2026 (evaluated once)

In [ ]:
final = lgbm_from(best_params).fit(dev[FEATURES], dev[TARGET])
base = make_models()["Grid-only logistic (baseline)"][1].fit(dev[["grid_pos"]], dev[TARGET])
p_test = final.predict_proba(test[FEATURES])[:, 1]
p_base = base.predict_proba(test[["grid_pos"]])[:, 1]
test_metrics = {"model": metrics(test, test[TARGET].values, p_test),
                "grid_only_baseline": metrics(test, test[TARGET].values, p_base),
                "rule_top3_grid": {"podium_hits_per_race": float(race_top3_hits(test, -test.grid_pos.values).mean()),
                                   "winner_hit_rate": race_winner_hit(test, -test.grid_pos.values)}}
by_year = []
for y in TEST_YEARS:
    m = test.year == y
    by_year.append({"year": y, "races": int(test[m].raceId.nunique()),
                    "model_log_loss": metrics(test[m], test[m][TARGET].values, p_test[m])["log_loss"],
                    "baseline_log_loss": metrics(test[m], test[m][TARGET].values, p_base[m])["log_loss"],
                    "model_podium_hits": float(race_top3_hits(test[m], p_test[m]).mean()),
                    "grid_rule_podium_hits": float(race_top3_hits(test[m], -test[m].grid_pos.values).mean())})
by_year = pd.DataFrame(by_year)
display(pd.DataFrame(test_metrics).T.round(4)); by_year.round(3)

In [ ]:
ll_gain = 1 - test_metrics["model"]["log_loss"] / test_metrics["grid_only_baseline"]["log_loss"]
print(f"Log-loss improvement over the grid-only model on 2023–2026: {ll_gain:.1%}")
facts["test_logloss_improvement"] = round(float(ll_gain), 4)

### 4.1 Are the probabilities honest? (calibration)

In [ ]:
frac, mean_p = calibration_curve(test[TARGET], p_test, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(5.6, 5.4))
ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
ax.plot(mean_p, frac, color=BLUE, marker="o", markersize=7)
ax.set_xlabel("Predicted podium probability"); ax.set_ylabel("Observed podium rate")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("When it says 60%, it happens ~60%")
style.subtitle(ax, "Calibration on the 2023–2026 test set (10 equal-size bins)")
style.source(fig); plt.savefig(FIG + "08_calibration.png"); plt.show()

### 4.2 What drives a podium? (SHAP on the test set)

In [ ]:
explainer = shap.TreeExplainer(final)
sv = explainer.shap_values(test[FEATURES])
sv = sv[1] if isinstance(sv, list) else sv
imp = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES).sort_values(ascending=False)
facts["shap_top"] = imp.head(8).round(4).to_dict()
top = imp.head(10)[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top.index, top.values, color=[ORANGE if i >= len(top) - 3 else BLUE for i in range(len(top))], height=0.6)
ax.grid(axis="y", visible=False); ax.set_xlabel("mean |SHAP| (impact on log-odds)")
ax.set_title("Grid first — then the team's recent form")
style.subtitle(ax, "Top 10 features by mean absolute SHAP value, 2023–2026 test set")
style.source(fig); plt.savefig(FIG + "09_shap_importance.png"); plt.show()

In [ ]:
with plt.style.context("default"):
    shap.summary_plot(sv, test[FEATURES], max_display=12, show=False, plot_size=(9, 6))
    plt.title("How each feature pushes the podium probability (test set)", loc="left", fontweight="bold")
    plt.savefig(FIG + "10_shap_beeswarm.png", dpi=150, bbox_inches="tight"); plt.show()

### 4.3 Case studies — races the grid got wrong

In [ ]:
tp = test[["raceId", "year", "round", "name", "driverId", "constructorId", "grid_pos", "positionOrder", "podium"]].copy()
tp["p_model"], tp["p_grid_baseline"] = p_test, p_base
tp = tp.merge(t["drivers"][["driverId", "forename", "surname", "code"]], on="driverId") \
       .merge(t["constructors"][["constructorId", "name"]].rename(columns={"name": "team"}), on="constructorId")
tp.to_csv(RES + "test_predictions.csv", index=False)
# podium finishers who started outside the top 5: who did the model rate higher than the grid did?
surprise = tp[(tp.podium == 1) & (tp.grid_pos > 5)].copy()
surprise["lift"] = surprise.p_model / surprise.p_grid_baseline
facts["surprise_podiums"] = int(len(surprise))
facts["surprise_model_higher_share"] = round(float((surprise.p_model > surprise.p_grid_baseline).mean()), 3)
print(f"{len(surprise)} podiums from P6 or lower; the model gave a higher chance than the grid-only model in "
      f"{facts['surprise_model_higher_share']:.0%} of them")
surprise.sort_values("grid_pos", ascending=False)[["year", "name", "surname", "team", "grid_pos", "positionOrder", "p_grid_baseline", "p_model"]].head(10).round(3)

In [ ]:
case = tp[(tp.year == 2026) & (tp.name == "Italian Grand Prix")].sort_values("p_model", ascending=False)
facts["case_monza_2026"] = case.head(6)[["surname", "team", "grid_pos", "positionOrder", "p_model", "p_grid_baseline"]].round(3).to_dict(orient="records")
fig, ax = plt.subplots(figsize=(10, 5))
d = case.head(10)[::-1]
c = [ORANGE if p == 1 else BLUE for p in d.podium]
ax.barh(d.surname + "  (P" + d.grid_pos.astype(int).astype(str) + " start)", d.p_model * 100, color=c, height=0.6)
for i, (pm, fin) in enumerate(zip(d.p_model, d.positionOrder)):
    ax.text(pm * 100 + 1, i, f"{pm:.0%}  → finished P{fin}", va="center", color=INK_2, fontsize=9.5)
ax.xaxis.set_major_formatter(mtick.PercentFormatter()); ax.set_xlim(0, 105); ax.grid(axis="y", visible=False)
ant = case[case.surname == "Antonelli"].iloc[0]
facts["case_antonelli"] = {"grid": int(ant.grid_pos), "p_model": round(float(ant.p_model), 4), "p_grid_baseline": round(float(ant.p_grid_baseline), 4)}
ax.set_title(f"Monza 2026: Antonelli won from P{int(ant.grid_pos)} — the model still gave him {ant.p_model:.0%}")
style.subtitle(ax, f"Pre-race podium odds (grid-only model: {ant.p_grid_baseline:.2%} for him) · orange = finished on the podium")
style.source(fig); plt.savefig(FIG + "11_case_monza_2026.png"); plt.show()

## 5. Save everything

In [ ]:
# production model: same tuned settings, trained on ALL seasons (used by the Streamlit what-if tab)
prod = lgbm_from(best_params).fit(df[FEATURES], df[TARGET])
joblib.dump({"model": prod, "features": FEATURES, "params": best_params}, "/kaggle/working/models/podium_lgbm.joblib")
out = {"facts": facts, "cv_table": cv_table.round(4).to_dict(orient="records"),
       "cv_folds_tuned": tuned_cv.round(4).to_dict(orient="records"), "best_params": best_params,
       "test": test_metrics, "test_by_year": by_year.round(4).to_dict(orient="records"),
       "features": FEATURES}
json.dump(out, open(RES + "metrics.json", "w"), indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
df.to_csv("/kaggle/working/data/model_table.csv", index=False)
print("saved")

## 6. Limitations & next steps
* Pre-race model: it cannot know about crashes, safety cars or rain — the reason even a perfect model misses podiums.
* 2026 is a new rules era; the model has only 2026's own previous rounds to learn the new pecking order (rolling form features adapt race by race).
* Qualifying 1994–2002 is incomplete in the source, so the model starts in 2006.
* Next: add practice-session pace (available only 2025–26), weather data, and a race-simulation of the championship.